<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Why layers need activations

**[Why layers need activations](https://learning.nextia-ai.com/courses/deep-learning/m01/activations/)** · Deep Learning and Transformers Explained · Module 1, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** show that layers without an activation collapse into one straight line, train small networks on 2-D toy data and draw their decision boundaries, and compute a tiny network with ReLU by hand.

| | |
|---|---|
| **Time** | About 20 minutes |
| **Needs** | No account and no data download: scikit-learn makes the toy data. The first cell checks PyTorch; Colab and Kaggle have it. |
| **You should know** | A neuron, a layer and how to count parameters, from [Neurons and layers](https://learning.nextia-ai.com/courses/deep-learning/m01/neurons-and-layers/). |
| **You do not need** | Any earlier notebook of this course. |
| **Tested** | 2026-10-08, Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled. The networks use scikit-learn 1.9.1; another version can give slightly different accuracies |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m01/activations/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M01-L03-activations/activations-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: the toy data and two helpers

The next cell makes the two toy datasets of the lesson's explorers, each with 200 training points and 200 new points. It also defines two helpers:

- `train_network(X, y, hidden, activation)` trains a network with one hidden layer, as in the explorer. `activation` is `"identity"` (no activation) or `"relu"`. With `hidden=0`, the network is one neuron.
- `draw_boundary(network, X, y, title)` shades where the network says each group and draws the boundary.

Run the cell. It prints nothing.

In [ ]:
import warnings
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles, make_moons
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore", category=ConvergenceWarning)

rings_X, rings_y = make_circles(200, noise=0.12, factor=0.45, random_state=0)
rings_new_X, rings_new_y = make_circles(200, noise=0.12, factor=0.45, random_state=1)
moons_X, moons_y = make_moons(200, noise=0.25, random_state=0)
moons_new_X, moons_new_y = make_moons(200, noise=0.25, random_state=1)

def train_network(X, y, hidden, activation):
    """One hidden layer of `hidden` neurons (none if 0), a logistic output neuron."""
    layers = (hidden,) if hidden else ()
    return MLPClassifier(layers, activation=activation, solver="lbfgs", max_iter=5000, random_state=0).fit(X, y)

def draw_boundary(network, X, y, title):
    """Shade the network's answer on a grid, draw the 50:50 line and the points."""
    xs = np.linspace(X[:, 0].min() - 0.3, X[:, 0].max() + 0.3, 200)
    ys = np.linspace(X[:, 1].min() - 0.3, X[:, 1].max() + 0.3, 200)
    gx, gy = np.meshgrid(xs, ys)
    p = network.predict_proba(np.c_[gx.ravel(), gy.ravel()])[:, 1].reshape(gx.shape)
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    ax.contourf(gx, gy, p, levels=[0, 0.5, 1], colors=["#dbe8fb", "#fbe2d6"])
    ax.contour(gx, gy, p, levels=[0.5], colors="#0B1A3F", linewidths=2)
    ax.scatter(*X[y == 0].T, marker="o", s=14, c="#3987e5", label="group A")
    ax.scatter(*X[y == 1].T, marker="^", s=18, c="#d95926", label="group B")
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.legend(loc="lower right", fontsize=8)
    plt.show()

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Two layers without an activation collapse

The smallest network: one input `x`, a hidden neuron h = 2x + 1, and an output neuron y = 3h − 4. Neither has an activation.

> **Predict.** Put h into y. Which single straight line do you get? Then run the cell: it compares the two layers with the line 6x − 1.

In [ ]:
def two_layers(x):
    h = 2 * x + 1
    return 3 * h - 4

for x in [-2, -1, -0.5, 0, 1, 2]:
    print(f"x = {x:>4}: two layers give {two_layers(x):>5}, the line 6x - 1 gives {6 * x - 1:>5}")

> **Check.** The two columns are the same for every x: 3 × (2x + 1) − 4 = 6x − 1. Two layers with four parameters are one straight line.

Now put ReLU, max(0, z), after the hidden neuron.

> **Your turn.** Write `two_layers_relu(x)`: the same network, but h = max(0, 2x + 1). Then run the check cell. It compares your outputs for x = −2, −1, −0.5, 0, 1 and 2 with the expected ones.

In [ ]:
def two_layers_relu(x):
    h = max(0, 2 * x + 1)
    return 3 * h - 4

print([two_layers_relu(x) for x in [-2, -1, -0.5, 0, 1, 2]])

In [ ]:
try:
    your_relu_outputs = [float(two_layers_relu(x)) for x in [-2, -1, -0.5, 0, 1, 2]]
except TypeError:  # the cell above still has ...
    your_relu_outputs = None

In [ ]:
expect_hash('your_relu_outputs', your_relu_outputs, '5650f6379b0dc4ac')

Left of x = −0.5 the output stays at −4, right of it it rises by 6 per step: two straight pieces and one bend. The ReLU made the bend.

## 2. The rings: no activation

The rings data has group B (triangles) in a disc and group A (circles) in a ring around it.

> **Predict.** A network with 16 hidden neurons and **no** activation: will its boundary bend around the disc? Write your prediction, then run the cell.

In [ ]:
network = train_network(rings_X, rings_y, 16, "identity")
print("training points right:", network.score(rings_X, rings_y))
draw_boundary(network, rings_X, rings_y, "16 hidden neurons, no activation")

> **Check.** You should see `training points right: 0.51` and one straight line across the disc: a coin toss.

## 3. The rings: ReLU

The next cell trains networks with 0, 2, 4, 8 and 16 hidden neurons, without and with ReLU, and prints the share of the training points and of the 200 new points that each gets right. Then it draws the network with 4 ReLU neurons.

Run the cell. It takes a few seconds.

In [ ]:
print("hidden  activation  training  new points")
for activation in ["identity", "relu"]:
    for hidden in [0, 2, 4, 8, 16]:
        network = train_network(rings_X, rings_y, hidden, activation)
        print(f"{hidden:>6}  {activation:>10}  {network.score(rings_X, rings_y):>8.3f}  {network.score(rings_new_X, rings_new_y):>10.3f}")
draw_boundary(train_network(rings_X, rings_y, 4, "relu"), rings_X, rings_y, "4 hidden neurons, ReLU")

> **Check.** Without an activation, every row is about 0.51. With ReLU: 0.66 for 2 neurons, 0.975 for 4, 0.99 for 8 and 1.0 for 16. The drawing shows a four-sided shape: four ReLU neurons, four straight pieces.

## 4. Change one thing: the moons, and the size to choose

The moons overlap a little, so no boundary is perfect, and too many bends start to follow noise.

> **Your turn.** For 2, 4, 8 and 16 ReLU neurons on the moons, compute the share of the **new** points right (`moons_new_X`, `moons_new_y`). Put the number of neurons with the highest share in `best_size`. Then run the check.

In [ ]:
new_scores = {}
for hidden in [2, 4, 8, 16]:
    network = train_network(moons_X, moons_y, hidden, "relu")
    new_scores[hidden] = network.score(moons_new_X, moons_new_y)
    print(hidden, "neurons: training", network.score(moons_X, moons_y), "| new points", new_scores[hidden])
best_size = max(new_scores, key=new_scores.get)
print("best size:", best_size)

In [ ]:
expect_hash('best_size', best_size, '2c624232cdd22177')

Sixteen neurons get the most training points right (0.97) but fewer new points (0.905) than 8 neurons (0.94). Their extra bends follow chance details of the 200 training points: overfitting. Draw both to see the spikes.

Run the cell.

In [ ]:
for hidden in [8, 16]:
    draw_boundary(train_network(moons_X, moons_y, hidden, "relu"), moons_X, moons_y, f"{hidden} hidden neurons, ReLU")

## 5. Failure case: a deep network without activations

Tomás builds three hidden layers of 64 neurons each and forgets the activations.

> **Predict.** With 8,577 parameters, will it separate the rings? Then run the cell.

In [ ]:
for activation in ["identity", "relu"]:
    deep = MLPClassifier((64, 64, 64), activation=activation, solver="lbfgs", max_iter=5000, random_state=0).fit(rings_X, rings_y)
    n_parameters = sum(w.size for w in deep.coefs_) + sum(b.size for b in deep.intercepts_)
    print(f"{activation:>8}: {n_parameters} parameters, training points right {deep.score(rings_X, rings_y):.3f}")

> **Check.** Without activations: about 0.5, no better than one neuron. With ReLU: 1.000. Layers without activations collapse into one weighted sum, however many there are. Put an activation after every hidden layer.

## 6. Module practice: a tiny network by hand

The network of the lesson's module practice: 2 inputs; hidden neuron 1 has the weights 2 and −1 and the bias 0; hidden neuron 2 has the weights −1 and 1 and the bias −2; both use ReLU. The output neuron has the weights 1.5 and −1 and the bias −1, then the logistic function.

Run the cell. It prints every intermediate number for the input (2, 1).

In [ ]:
import math

def tiny_network(x1, x2, show=True):
    h1_raw = 2 * x1 - 1 * x2 + 0
    h2_raw = -1 * x1 + 1 * x2 - 2
    h1, h2 = max(0, h1_raw), max(0, h2_raw)
    z = 1.5 * h1 - 1 * h2 - 1
    score = 1 / (1 + math.exp(-z))
    if show:
        print(f"hidden before ReLU ({h1_raw}, {h2_raw}), after ReLU ({h1}, {h2}); z = {z}; score = {score:.3f}")
    return score

tiny_network(2, 1);

> **Check.** You should see `hidden before ReLU (3, -3), after ReLU (3, 0); z = 3.5; score = 0.971`. ReLU switched hidden neuron 2 off.

> **Your turn.** First compute the network **by hand** for the input (0, 3): both hidden neurons before and after ReLU, then z, then the score. Put your hand-computed score, rounded to 3 decimals, in `my_score`. Then run the check, and run `tiny_network(0, 3)` to see every step.

In [ ]:
my_score = 0.119  # h = (0, 1) after ReLU, z = -2, 1 / (1 + e^2)

In [ ]:
expect('my_score', my_score, 0.119, tolerance=0.001)

In [ ]:
tiny_network(0, 3);

## Next

You can now show why layers without activations add no power, see ReLU bend a boundary, and choose a network's size with new data. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m01/activations/) has the explorers, the module check and the knowledge checks that count toward your certificate. Module 2 turns these pictures into PyTorch code, starting with tensors and their shapes.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Tensors and shapes](https://learning.nextia-ai.com/courses/deep-learning/m02/tensors-and-shapes/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m01/activations/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The toy data is generated by scikit-learn and has no real people.